# Learning with constraints: forward-backward and EM

`baum_welch_mvr_chmm` maximizes

$$\log P\big(y,\ \text{constraints satisfied}\ \big|\ \theta\big)$$

The generative model is the *unconstrained* HMM. Feasibility is an
observed fact about each realized hidden path: ie. the data was drawn from the unconstrained model, and we happen to know that the latent path satisfied `constraints`.  

The complete-data likelihood is
$P(x, y \mid \theta)\,\mathbb{1}[x \text{ feasible}]$, and maximizing $E_q[\log P(x, y \mid \theta)]$ stays the ordinary
weighted-count normalization. This is therefore **exact, closed-form, monotone
EM** with no partition-function term.

A $Z(\theta) = P(\text{feasible} \mid \theta)$ correction would only enter if the
data had been drawn from the *renormalized* constrained distribution — **this will be implemented at a later time**.

In [ ]:
import itertools
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.learning.baum_welch_mvr import (
    baum_welch_mvr_chmm,
    forward_backward_mvr_chmm,
)

## 1. The model

The same three-state HMM as the inference notebooks. The constraint is a **subsequence** one: `C` is forbidden over time `[0, 3]`. A global constraint that bans `C` is equivalent to a restriction on the parameter space: we use a subsequence constraint to show how path constraint information can be integrated into learning.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.2765440507007986,
    "B": 0.4033576072467887,
    "C": 0.32009834205241255,
}

transition_probs = {
    ("A", "A"): 0.3391777054270445,
    ("A", "B"): 0.049711711669595204,
    ("A", "C"): 0.6111105829033604,
    ("B", "A"): 0.48102507253852517,
    ("B", "B"): 0.05601918704283972,
    ("B", "C"): 0.4629557404186351,
    ("C", "A"): 0.43616112524444134,
    ("C", "B"): 0.1773076392327265,
    ("C", "C"): 0.38653123552283214,
}

emission_probs = {
    ("A", "lo"): 0.19949219710155375,
    ("A", "mid"): 0.30789837305397333,
    ("A", "hi"): 0.492609429844473,
    ("B", "lo"): 0.534907622618408,
    ("B", "mid"): 0.234417585356662,
    ("B", "hi"): 0.23067479202493,
    ("C", "lo"): 0.09093879934300991,
    ("C", "mid"): 0.008996844382398088,
    ("C", "hi"): 0.9000643562745919,
}

true_hmm = HiddenMarkovModel()
true_hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

T = 8
WINDOW = [0, 3]


def forbid_mvr(state, time_range=None, name=None):
    """MVR rejecting any path that visits ``state`` inside its window."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
        name=name,
    )


no_C_early = forbid_mvr("C", time_range=WINDOW, name="no_C_early")
print(f"constraint: no C at t in [{WINDOW[0]}, {WINDOW[1]}], horizon T = {T}")

## 2. Forward-backward

`forward_backward_mvr_chmm` returns the posterior over hidden states given the
observations **and** the fact that the path is feasible. The window shows up
directly: `P(C)` is exactly zero for `t <= 3` and free afterwards.

In [ ]:
rng = np.random.default_rng(3)
repn = true_hmm.repn


def sample_constrained(n):
    """Rejection-sample sequences whose hidden path satisfies the constraint."""
    out, tries = [], 0
    while len(out) < n:
        tries += 1
        hidden = [rng.choice(3, p=repn.start_vec)]
        for _ in range(T - 1):
            hidden.append(rng.choice(3, p=repn.transition_mat[hidden[-1]]))
        if 2 in hidden[: WINDOW[1] + 1]:
            continue
        out.append([
            OBSERVED_STATES[rng.choice(3, p=repn.emission_mat[h])] for h in hidden
        ])
    return out, tries


sequences, tries = sample_constrained(80)
print(f"{len(sequences)} feasible sequences kept from {tries} draws "
      f"({len(sequences) / tries:.1%} acceptance)")

truth = MVR_CHMM(hidden_markov_model=true_hmm, constraints=[no_C_early])
gamma, xi, loglik = forward_backward_mvr_chmm(truth, sequences[0])

print(f"\nloglik = {loglik:.6f}   gamma {tuple(gamma.shape)}   xi {tuple(xi.shape)}")
print()
print(pd.DataFrame(
    np.round(gamma.numpy(), 4), columns=HIDDEN_STATES,
).rename_axis("t").to_string())

### Checked against enumeration

$3^8 = 6561$ paths, so the posterior can be computed by brute-force.

In [ ]:
def brute_force_posterior(obs):
    """Exact gamma and loglik by enumerating every feasible hidden path."""
    idx_of = hmm_idx = true_hmm.hidden_to_internal
    r = true_hmm.repn
    total, marg = 0.0, np.zeros((T, 3))

    for path in itertools.product(range(3), repeat=T):
        if 2 in path[: WINDOW[1] + 1]:
            continue
        w = r.start_vec[path[0]]
        for t in range(1, T):
            w *= r.transition_mat[path[t - 1]][path[t]]
        for t, o in enumerate(obs):
            w *= r.emission_mat[path[t]][true_hmm.observed_to_internal[o]]
        total += w
        for t in range(T):
            marg[t, path[t]] += w

    return marg / total, math.log(total)


exact_gamma, exact_loglik = brute_force_posterior(sequences[0])

print(f"loglik  algorithm {loglik:.10f}   brute force {exact_loglik:.10f}")
print(f"gamma   max abs diff {np.abs(exact_gamma - gamma.numpy()).max():.3e}")
print(f"\nP(C) at t <= {WINDOW[1]}: {np.round(gamma.numpy()[: WINDOW[1] + 1, 2], 12)}")
print(f"P(C) at t >  {WINDOW[1]}: {np.round(gamma.numpy()[WINDOW[1] + 1 :, 2], 4)}")

## 3. EM: Constraints as Observations vs Constrained Learning
We demonstrate the distinction between 1. learning with constraints as additional information about the unconstrained latent path 2. learning a constrained distribution.
For example, consider a constraint that forbids visits to `C`. This can be interpreted as:

1. We drew a run from the unconstrained chain, and our draw didn't visit `C` by chance. This is given as additional information about the latent path.
2. We drew a run from the constrained chain where visits to `C` were forbidden. This is a constrained distribution.


In this experiment, runs are drawn from the constrained model $P(Y \ \ \big| \ \text{constraints satisfied}, \ \theta )$ but are incorporated as observation from the unconstrained model $P(Y, \ \text{constraints satisfied} \ \big| \ \theta )$. Our model is misspecified. Nonetheless, since EM is known to be monotone, the log-likelihood must increase every iteration even in this misspecified scenario. 

In [ ]:
def flat_hmm():
    """A deliberately uninformative starting point, with no structural zeros."""
    rows = [[0.5, 0.3, 0.2], [0.2, 0.5, 0.3], [0.3, 0.2, 0.5]]
    model = HiddenMarkovModel()
    model.load_model(
        start_probs={h: 1 / 3 for h in HIDDEN_STATES},
        transition_probs={
            (a, b): rows[i][j]
            for i, a in enumerate(HIDDEN_STATES)
            for j, b in enumerate(HIDDEN_STATES)
        },
        emission_probs={
            (h, o): rows[i][j]
            for i, h in enumerate(HIDDEN_STATES)
            for j, o in enumerate(OBSERVED_STATES)
        },
        initialize=True,
    )
    return model


start_model = MVR_CHMM(hidden_markov_model=flat_hmm(), constraints=[no_C_early])
fitted, history = baum_welch_mvr_chmm(start_model, sequences, max_iter=200, tol=1e-9)

steps = np.diff(history)
print(f"{len(history)} iterations")
print(f"loglik {history[0]:.4f} -> {history[-1]:.4f}")
print(f"monotone: {bool((steps >= -1e-9).all())}   smallest step {steps.min():.2e}")

### About that warning

EM ran out of its iteration budget before the change dropped below `tol=1e-9`, so
it warns:

- `history[i]` is the log-likelihood at the **start** of iteration `i`, before that
  iteration's update. So `history[0]` always scores the model passed in.
- If EM *converges*, no update follows the last entry and `history[-1]` scores the
  returned model exactly.
- If it stops at `max_iter` instead — as here — one further update was applied, so
  the returned model is one step **ahead** of `history[-1]`. You can see this
  below: the fitted model scores slightly better than `history[-1]` reports.

Passing `tol=0` will automatically make EM run for `max_iter` iterations and
suppresses the warning.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history, color="#2E6DB4", linewidth=2)
ax1.set_xlabel("EM iteration")
ax1.set_ylabel(r"$\log P(y,\ \mathrm{constraints}\ |\ \theta)$")
ax1.set_title("Constrained log-likelihood", loc="left", fontsize=11)
ax1.grid(color="0.9", linewidth=0.8)
ax1.set_axisbelow(True)

ax2.semilogy(np.maximum(steps, 1e-16), color="#E08A3C", linewidth=2)
ax2.set_xlabel("EM iteration")
ax2.set_ylabel("increase per iteration")
ax2.set_title("Every step is an increase", loc="left", fontsize=11)
ax2.grid(color="0.9", linewidth=0.8)
ax2.set_axisbelow(True)

fig.tight_layout()
plt.show()

## 4. What EM actually optimizes, and what it does not

Fitting the same sequences **without** the constraint optimizes
$\log P(y \mid \theta)$ rather than $\log P(y, \text{constraints} \mid \theta)$, so scoring both on the constrained objective shows what the constraint bought. Note again both are still assuming a misspecified unconstrained model, where as the true data-generating distribution was constrained. Nonetheless, incorporating the constraint still improves the log-likelihood.

The last row is the one to look at twice.

In [ ]:
unconstrained = MVR_CHMM(hidden_markov_model=flat_hmm(), constraints=[])
free_fit, _ = baum_welch_mvr_chmm(unconstrained, sequences, max_iter=200, tol=1e-9)


def constrained_loglik(model):
    """Total log P(y, constraints | theta) over the whole data set."""
    chmm = MVR_CHMM(hidden_markov_model=model, constraints=[no_C_early])
    return sum(forward_backward_mvr_chmm(chmm, s)[2] for s in sequences)


rows = [
    ("starting model", flat_hmm()),
    ("unconstrained fit", free_fit),
    ("constrained fit", fitted),
    ("true model", true_hmm),
]

print(pd.DataFrame(
    [(name, round(constrained_loglik(m), 3)) for name, m in rows],
    columns=["model", "log P(y, constraints | theta)"],
).to_string(index=False))

print(f"\nhistory[0]      = {history[0]:.3f}")
print(f"starting model  = {constrained_loglik(flat_hmm()):.3f}   (history[0] scores the model passed in)")

**The true model scores worst.** Maximum likelihood returns the parameters that best explain the data,
not the parameters that generated it. With 80 sequences of length 8 and emissions
that separate the three states only weakly, EM finds a sharper, more degenerate
explanation that fits the sample better than the true model.

This notebook does **not** claim parameter recovery. For example, the fitted emission matrix is not close to that of `true_hmm`, and
the start vector has collapsed onto a single state. This just points to common pitfalls of EM with weakly identified models:

- the likelihood is multimodal, and EM finds a local optimum determined its intialization;
- hidden-state labels are arbitrary, so a fit is comparable to a reference only
  up to a permutation of the states.

What *is* guaranteed here is the monotone climb and the ordering
above — the constrained fit beats every other model on the objective it optimizes.

In [ ]:
print("Fitted vs true emissions — deliberately not close:\n")
print(pd.concat([
    pd.DataFrame(np.round(np.asarray(true_hmm.emission_mat, dtype=float), 3),
                 index=HIDDEN_STATES, columns=OBSERVED_STATES).add_prefix("true: "),
    pd.DataFrame(np.round(np.asarray(fitted.emission_mat, dtype=float), 3),
                 index=HIDDEN_STATES, columns=OBSERVED_STATES).add_prefix("fit: "),
], axis=1).to_string())

print("\nstart vector")
print(f"  true {np.round(np.asarray(true_hmm.start_vec, dtype=float), 3)}")
print(f"  fit  {np.round(np.asarray(fitted.start_vec, dtype=float), 3)}   <- collapsed")

## Notes

- The objective is $\log P(y, \text{constraints} \mid \theta)$. The constraint
  enters the E-step — counts are taken under the posterior restricted to feasible
  paths — and the M-step is then the ordinary normalization of those counts.
- `history[i]` is the log-likelihood at the **start** of iteration `i`, before
  that iteration's update, so `history[0]` scores the model passed in. If EM
  converged, `history[-1]` scores the returned model exactly; if it stopped at
  `max_iter` instead, one further update was applied and the returned model is a
  step ahead of `history[-1]`. That case warns, unless `tol <= 0`, which is an
  explicit request for exactly `max_iter` iterations.
- The fit runs on a `deepcopy`, so the model passed in is untouched and its
  constraint alignment survives.
- Parameters are written back in place rather than through `load_model`, which
  would re-derive the state ordering from sorted label names and could silently
  permute the internal indices.
- Emission counts accumulate only at observed times. An unobserved time still
  drives the chain and every MVR active at it, but contributes no emission
  statistic.
- `dtype` defaults to `float64` here, unlike Viterbi's `float32`: this recursion
  carries constraint-satisfaction probabilities whose spread grows with the
  horizon.
- Hidden-state labels are arbitrary under EM. Compare a fit to a reference only
  up to a permutation of the states.
- EM is a local method on a multimodal likelihood, and maximum likelihood on a
  finite sample does not return the generating parameters. Section 4 shows the
  true model scoring *worse* than the fit on the very objective being maximized.
  Judge a fit by its objective, not by its distance from parameters you happen to
  know.